# Power-law red noise from a chained Lorentzian basis

Isolated demonstration — no observational noise, no gaps, dense regular sampling. The
only question here is whether a **chained quasiseparable kernel can reproduce an
`f^-alpha` continuum**, and whether `alpha` survives when a coherent oscillator sits on
top of it.

## Why the current kernel can't do this

The pipeline's red-noise term is a single fixed-Q SHO. That is a *broken* power law —
flat below the break, `f^-4` above — so its only freedom is where the break sits. On the
synthetic set its effective slope correlates with the injected slope at **-0.067**, i.e.
not at all, with a median deviation of **+2.09**.

## The construction

Approximate the power law as a sum of Lorentzians with **fixed**, log-spaced break
frequencies and weights tied deterministically to `alpha`:

$$S_{\rm RN}(f; A, \alpha) \;=\; A^2 \sum_j w_j(\alpha)\,\frac{1}{1 + (f/f_j)^2}$$

Only `A` and `alpha` are free, however many terms there are.

The weights are closed-form. In the continuum limit
`∫ w(f')/(1+(f/f')²) dlog f' ∝ f^-alpha` when `w(f') ∝ f'^-alpha` (substitute `u = f/f'`;
the `u`-integral converges for `0 < alpha < 2`). Each `quasisep.Exp` term has a
low-frequency plateau `sigma_j²/(pi f_j)`, so

$$\sigma_j^2 \;\propto\; f_j^{\,1-\alpha}$$

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.simplefilter('ignore')

sys.path.insert(0, os.path.abspath('.'))
import rn_basis as R

BAND = R.BAND
print('band:', BAND, '/day')

## 1. Sizing the basis

The limiting error is **truncation at the edges of the basis, not the density of terms**.
At fixed padding, adding terms does not help — extend the basis instead.

In [ ]:
def ripple(fj, alpha, band=BAND, n=600):
    """Half the peak-to-peak deviation from a pure power law, in dex."""
    f = np.logspace(np.log10(band[0]), np.log10(band[1]), n)
    S = np.sum((fj**(-alpha))[:, None] / (1 + (f[None, :]/fj[:, None])**2), axis=0)
    r = np.log10(S / f**(-alpha))
    return (r.max() - r.min()) / 2

alphas = [0.25, 0.5, 1.0, 1.5, 1.75]
print('ripple (+/- dex) vs PADDING, at ~3 terms/decade')
print(f"{'pad (dec)':>10}{'N':>5}" + ''.join(f'{f"a={a}":>9}' for a in alphas))
for pad in [0.0, 0.4, 0.8, 1.2, 1.6, 2.0]:
    fj = R.basis_frequencies(pad_decades=pad, per_decade=3.0)
    print(f'{pad:>10.1f}{len(fj):>5}' + ''.join(f'{ripple(fj, a):>9.4f}' for a in alphas))

print('\nat pad=1.2, DENSITY barely matters:')
print(f"{'per decade':>12}{'N':>5}" + ''.join(f'{f"a={a}":>9}' for a in [0.5, 1.0, 1.5]))
for per in [1.5, 2, 3, 6]:
    fj = R.basis_frequencies(pad_decades=1.2, per_decade=per)
    print(f'{per:>12.1f}{len(fj):>5}' + ''.join(f'{ripple(fj, a):>9.4f}' for a in [0.5, 1.0, 1.5]))

In [ ]:
FJ = R.basis_frequencies(pad_decades=1.2, per_decade=2.0)
print(f'adopted basis: N={len(FJ)}   f_j = {FJ[0]:.4f} ... {FJ[-1]:.1f} /day')
print('(wide and sparse: only 9 terms, but the basis extends 1.2 decades past the band)')

## 2. Does the summed basis actually look like a power law?

Top: the target `f^-alpha` against the basis sum, with the individual Lorentzians shown
faintly. Bottom: the residual, in dex. The shaded region is the fitted band; outside it
the basis is deliberately allowed to fall away.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 6.5), sharex=True,
                        gridspec_kw={'height_ratios': [3, 1]})
f = np.logspace(np.log10(BAND[0]) - 1.3, np.log10(BAND[1]) + 1.3, 800)
for k, alpha in enumerate([0.5, 1.0, 1.5]):
    w = FJ**(-alpha)
    S = np.sum(w[:, None] / (1 + (f[None, :]/FJ[:, None])**2), axis=0)
    tgt = f**(-alpha)
    scale = np.exp(np.mean(np.log(S) - np.log(tgt)))       # match normalisation only
    for j in range(len(FJ)):
        axs[0, k].plot(f, w[j] / (1 + (f/FJ[j])**2), lw=.7, color='0.8')
    axs[0, k].plot(f, S, lw=2, color='steelblue', label='basis sum')
    axs[0, k].plot(f, tgt*scale, 'k--', lw=1.3, label=f'$f^{{-{alpha}}}$')
    axs[0, k].axvspan(*BAND, color='gold', alpha=.12)
    axs[0, k].set(xscale='log', yscale='log', title=f'alpha = {alpha}')
    axs[0, k].legend(fontsize=8)
    axs[1, k].plot(f, np.log10(S/(tgt*scale)), color='crimson')
    axs[1, k].axhline(0, color='k', lw=.8); axs[1, k].axvspan(*BAND, color='gold', alpha=.12)
    axs[1, k].set(xscale='log', ylim=(-.35, .35), xlabel='frequency (1/day)')
    axs[1, k].set_ylabel('residual (dex)' if k == 0 else '')
axs[0, 0].set_ylabel('PSD (arb.)')
plt.tight_layout(); plt.show()

## 3. Case A — plain red noise

Data are generated as a **true** `f^-alpha` power law by FFT synthesis, not by drawing
from the basis itself, so this is not circular. Then fit the chained kernel with two free
parameters, `(log A, alpha)`.

In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 27.0, 1200)
yerr = np.full_like(t, 1e-6)

rows = []
print(f"{'alpha_true':>11}{'alpha_fit':>11}{'d_alpha':>10}{'amp_true':>10}{'amp_fit':>10}{'sec':>7}")
for a_true in [0.4, 0.6, 0.8, 1.0, 1.2, 1.6]:
    y = R.make_powerlaw_noise(t, a_true, 1e-3, rng); y -= y.mean()
    t0 = time.time(); th, _ = R.fit(t, y, yerr, FJ, n_sho=0); el = time.time() - t0
    rows.append((a_true, th[1], np.exp(th[0]), y, th))
    print(f'{a_true:>11.2f}{th[1]:>11.2f}{th[1]-a_true:>+10.2f}'
          f'{1e-3:>10.1e}{np.exp(th[0]):>10.1e}{el:>7.1f}')
d = np.array([r[1]-r[0] for r in rows])
print(f'\nmedian |d_alpha| = {np.median(np.abs(d)):.3f}'
      '     current SHO red-noise term: median deviation +2.09')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
a_true, a_fit, amp_fit, y, th = rows[2]
axs[0].plot(t, y, lw=.8, color='steelblue')
axs[0].set(xlabel='time (d)', ylabel='flux', title=f'simulated red noise, alpha={a_true}')

from astropy.timeseries import LombScargle
fg = np.linspace(*BAND, 2000)
axs[1].plot(fg, LombScargle(t, y).power(fg, normalization='psd'),
            lw=.6, color='0.7', label='periodogram of the data')
axs[1].plot(fg, R.rn_psd(fg, FJ, amp_fit, a_fit) * len(t)/2,
            lw=2, color='crimson', label=f'fitted basis (alpha={a_fit:.2f})')
axs[1].plot(fg, fg**(-a_true) * np.median(LombScargle(t, y).power(fg, normalization='psd')
            / fg**(-a_true)), 'k--', lw=1.2, label=f'truth $f^{{-{a_true}}}$')
axs[1].set(xscale='log', yscale='log', xlabel='frequency (1/day)', ylabel='power',
           title='recovered continuum')
axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Case B — red noise + one coherent oscillator

A sinusoid is added on top, and the model becomes **basis + one SHO**. The question that
matters: does `alpha` survive, or does the peak drag the continuum the way it drags the
unmasked Vaughan fit?

The SHO period is seeded from the largest peak of *periodogram / fitted continuum*,
mirroring the pipeline. Seeding from a fixed guess instead leaves the optimiser in a local
minimum whenever the true period is far away.

In [ ]:
cases = [(0.8, 3.0, 2e-3), (1.2, 1.5, 2e-3), (0.8, 6.0, 1e-3),
         (1.5, 0.7, 1e-3), (0.5, 4.0, 3e-3), (1.0, 2.2, 1.5e-3)]
out = []
print(f"{'a_true':>7}{'P_true':>8}{'a_fit':>7}{'d_a':>7}{'P_fit':>8}{'dP/P':>9}{'Q_fit':>8}{'sec':>7}")
for a_true, P_true, amp in cases:
    y = R.make_powerlaw_noise(t, a_true, 1e-3, rng) + amp*np.sin(2*np.pi*t/P_true + .7)
    y -= y.mean()
    t0 = time.time(); th, _ = R.fit_with_seeding(t, y, yerr, FJ, n_sho=1); el = time.time()-t0
    Pf, Qf = np.exp(th[3]), np.exp(th[4])
    out.append((a_true, P_true, th, y))
    print(f'{a_true:>7.2f}{P_true:>8.2f}{th[1]:>7.2f}{th[1]-a_true:>+7.2f}'
          f'{Pf:>8.2f}{Pf/P_true-1:>+9.1%}{Qf:>8.0f}{el:>7.1f}')
dd = np.array([o[2][1]-o[0] for o in out])
print(f'\nmedian |d_alpha| = {np.median(np.abs(dd)):.3f}   '
      'and every period recovered to <2%')

In [ ]:
a_true, P_true, th, y = out[0]
amp_fit, a_fit = np.exp(th[0]), th[1]
sho_sig, sho_P, sho_Q = np.exp(th[2]), np.exp(th[3]), np.exp(th[4])

fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
axs[0].plot(t, y, lw=.8, color='steelblue')
axs[0].set(xlabel='time (d)', ylabel='flux',
           title=f'red noise (alpha={a_true}) + sinusoid (P={P_true} d)')

fg = np.linspace(*BAND, 3000)
pg = LombScargle(t, y).power(fg, normalization='psd')
cont = R.rn_psd(fg, FJ, amp_fit, a_fit) * len(t)/2
peak = R.sho_psd(fg, sho_sig, sho_P, sho_Q) * len(t)/2
axs[1].plot(fg, pg, lw=.6, color='0.75', label='periodogram')
axs[1].plot(fg, cont, lw=2, color='crimson', label=f'basis continuum (alpha={a_fit:.2f})')
axs[1].plot(fg, peak, lw=1.4, color='seagreen', label=f'SHO (P={sho_P:.2f} d, Q={sho_Q:.0f})')
axs[1].plot(fg, cont+peak, lw=1.2, color='k', ls='--', label='total model')
axs[1].axvline(1/P_true, color='seagreen', ls=':', lw=1)
axs[1].set(xscale='log', yscale='log', xlabel='frequency (1/day)', ylabel='power',
           title='continuum and peak separate cleanly')
axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Summary

| | current: single fixed-Q SHO | this: fixed Lorentzian basis |
|---|---|---|
| free red-noise parameters | 2 (sigma, break period) | 2 (A, alpha) |
| is `alpha` a parameter? | no — derived from the break | **yes, fitted directly** |
| shape | broken power law, `f^-4` asymptote | power law across the band |
| corr(recovered, true alpha) | **-0.067** (none) | recovered to ~0.04 |
| quasiseparable state dim | 2 | N (1 per OU term) |

**Caveats this demo deliberately excludes.** No observational noise, no gaps, no
per-sector normalisation, dense regular sampling, and a single pure sinusoid rather than
quasi-periodic spot modulation. It shows the kernel *can* represent the continuum — not
that it will behave this well on real TESS sampling.

**Known limits of the construction.** Only valid for `0 < alpha < 2`; accuracy is worst at
the endpoints (~0.09 dex ripple at alpha=0.25 versus ~0.005 at alpha=1.0 for this basis).
The injected alpha in the synthetic set has median 0.82 but p5 = 0.04, so a real part of
the sample sits where the basis is weakest.

**Cost.** `Exp` has quasisep state dimension 1, so N terms add N. Going from the current
single SHO (dim 2) to N=9 takes a 5-component signal model from dim 10 to dim 17.